# Linear Regression using Gradient Descent with PyTorch Autograd

## CPU vs GPU vs `torch.compile` Performance Comparison

This notebook implements linear regression from scratch using PyTorch's automatic differentiation (`autograd`).
We train the same model on **CPU**, **GPU**, and a **`torch.compile`-optimized** version, then compare running times.

### Goals
1. Understand how PyTorch builds a **computation graph** dynamically.
2. Use `requires_grad=True` and `.backward()` to compute gradients automatically.
3. Manually implement the gradient-descent update step (so we *see* the math).
4. Benchmark training on CPU vs GPU.
5. Use `torch.compile` (PyTorch 2.x) to JIT-compile the training step and measure the additional speedup.

## 1. Imports and Setup

In [ ]:
import time
import torch
import numpy as np
import matplotlib.pyplot as plt

# For reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device     : {torch.cuda.get_device_name(0)}")

## 2. Generate Synthetic Data

We create a synthetic dataset that follows the linear relationship:

$$ y = X w_{true} + b_{true} + \epsilon $$

where $\epsilon$ is small Gaussian noise.

Using a **large** dataset makes the GPU advantage visible — on tiny tensors the GPU is actually slower because of data-transfer overhead.

In [ ]:
# Dataset size — feel free to scale these up/down
N_SAMPLES  = 100_000   # number of training examples
N_FEATURES = 500       # input dimension

# True parameters that we want our model to recover
true_w = torch.randn(N_FEATURES, 1)
true_b = torch.tensor([[2.5]])

# Inputs and noisy targets
X = torch.randn(N_SAMPLES, N_FEATURES)
noise = 0.1 * torch.randn(N_SAMPLES, 1)
y = X @ true_w + true_b + noise

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

## 3. Training Function with Autograd

We wrap the entire training loop in a function so we can run it on both devices with identical code.

### Key autograd ideas
- We create `w` and `b` with `requires_grad=True`. PyTorch will track every operation involving them.
- The forward pass `y_pred = X @ w + b` builds a **computation graph**.
- `loss.backward()` walks that graph in reverse and fills `w.grad` and `b.grad`.
- The update `w -= lr * w.grad` happens inside `torch.no_grad()` so it is NOT tracked.
- `.zero_()` on the gradients resets them — otherwise PyTorch accumulates.

In [ ]:
def train_linear_regression(X, y, device, n_epochs=200, lr=0.01, verbose=True):
    """Train linear regression with manual gradient descent using autograd.

    Parameters
    ----------
    X, y      : training tensors (will be moved to `device`)
    device    : 'cpu' or 'cuda'
    n_epochs  : number of full-batch gradient-descent steps
    lr        : learning rate

    Returns
    -------
    elapsed   : wall-clock training time in seconds
    losses    : list of loss values per epoch
    w, b      : learned parameters (still on `device`)
    """
    # Move data to the chosen device
    X_dev = X.to(device)
    y_dev = y.to(device)

    n_features = X_dev.shape[1]

    # Trainable parameters — autograd will track them
    w = torch.zeros(n_features, 1, device=device, requires_grad=True)
    b = torch.zeros(1, 1,         device=device, requires_grad=True)

    losses = []

    # Make sure all GPU work is finished before we start the timer
    if device == 'cuda':
        torch.cuda.synchronize()
    start = time.time()

    for epoch in range(n_epochs):
        # ---- Forward pass: builds the computation graph ----
        y_pred = X_dev @ w + b                       # linear model
        loss   = ((y_pred - y_dev) ** 2).mean()      # MSE loss

        # ---- Backward pass: autograd computes dL/dw and dL/db ----
        loss.backward()

        # ---- Manual gradient-descent update ----
        # Wrap in no_grad so the update itself isn't part of the graph
        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
            # Zero the gradients for the next iteration
            w.grad.zero_()
            b.grad.zero_()

        losses.append(loss.item())

        if verbose and (epoch + 1) % 50 == 0:
            print(f"  [{device:>4}] epoch {epoch+1:4d} | loss = {loss.item():.6f}")

    # Wait for any pending GPU work before stopping the timer
    if device == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.time() - start

    return elapsed, losses, w.detach(), b.detach()

## 4. Train on CPU

In [ ]:
print("Training on CPU...")
cpu_time, cpu_losses, w_cpu, b_cpu = train_linear_regression(
    X, y, device='cpu', n_epochs=200, lr=0.01
)
print(f"\nCPU training time: {cpu_time:.4f} seconds")
print(f"Final CPU loss   : {cpu_losses[-1]:.6f}")

## 5. Train on GPU

If no GPU is available we skip this cell and fall back to CPU-only results.

In [ ]:
if torch.cuda.is_available():
    # Warm-up run — the very first CUDA op pays a one-time initialization cost
    # that has nothing to do with our algorithm. We discard the timing.
    print("Warming up GPU...")
    _ = train_linear_regression(X, y, device='cuda', n_epochs=10, verbose=False)

    print("\nTraining on GPU...")
    gpu_time, gpu_losses, w_gpu, b_gpu = train_linear_regression(
        X, y, device='cuda', n_epochs=200, lr=0.01
    )
    print(f"\nGPU training time: {gpu_time:.4f} seconds")
    print(f"Final GPU loss   : {gpu_losses[-1]:.6f}")
else:
    gpu_time, gpu_losses = None, None
    print("CUDA not available — skipping GPU training.")

## 6. Train with `torch.compile`

`torch.compile` (introduced in PyTorch 2.0) traces the computation graph and JIT-compiles it into
fused, optimized kernels — often using **TorchInductor** as the backend.

Key things to know:
- `torch.compile(fn)` returns a wrapped function. The **first** call is *slow* because compilation happens then.
- Subsequent calls reuse the compiled kernels and are typically much faster, especially on GPU.
- Compile works on CPU too, but the wins are usually larger on GPU because more ops can be fused.

We refactor the per-step computation into a small function and wrap it with `torch.compile`.
We also do an explicit warm-up to exclude compilation time from the benchmark.

In [ ]:
def train_with_compile(X, y, device, n_epochs=200, lr=0.01, verbose=True):
    """Same training loop as before, but the per-step forward+backward is wrapped
    with torch.compile so PyTorch can JIT-compile it into fused kernels.
    """
    X_dev = X.to(device)
    y_dev = y.to(device)
    n_features = X_dev.shape[1]

    w = torch.zeros(n_features, 1, device=device, requires_grad=True)
    b = torch.zeros(1, 1,         device=device, requires_grad=True)

    # The function we want compiled: forward pass + loss.
    # We keep the .backward() call OUTSIDE because torch.compile composes
    # cleanly with autograd this way.
    def forward(X_dev, y_dev, w, b):
        y_pred = X_dev @ w + b
        loss   = ((y_pred - y_dev) ** 2).mean()
        return loss

    compiled_forward = torch.compile(forward)

    losses = []

    # ---- Warm-up: trigger compilation BEFORE we start timing ----
    # The first 1-2 calls compile the graph; we don't want that in the benchmark.
    for _ in range(3):
        loss = compiled_forward(X_dev, y_dev, w, b)
        loss.backward()
        with torch.no_grad():
            w.grad.zero_(); b.grad.zero_()
    # Reset parameters to zero so the warm-up didn't change our starting point
    with torch.no_grad():
        w.zero_(); b.zero_()

    if device == 'cuda':
        torch.cuda.synchronize()
    start = time.time()

    for epoch in range(n_epochs):
        loss = compiled_forward(X_dev, y_dev, w, b)
        loss.backward()

        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
            w.grad.zero_()
            b.grad.zero_()

        losses.append(loss.item())

        if verbose and (epoch + 1) % 50 == 0:
            print(f"  [compile/{device}] epoch {epoch+1:4d} | loss = {loss.item():.6f}")

    if device == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.time() - start

    return elapsed, losses, w.detach(), b.detach()

### 6a. `torch.compile` on CPU

In [ ]:
print("Training on CPU with torch.compile...")
try:
    cpu_compile_time, cpu_compile_losses, _, _ = train_with_compile(
        X, y, device='cpu', n_epochs=200, lr=0.01
    )
    print(f"\nCPU + torch.compile training time: {cpu_compile_time:.4f} seconds")
    print(f"Final loss: {cpu_compile_losses[-1]:.6f}")
except Exception as e:
    # torch.compile can fail on some platforms (e.g. Windows, older PyTorch).
    # We catch it so the rest of the notebook still runs.
    print(f"torch.compile on CPU failed: {e}")
    cpu_compile_time, cpu_compile_losses = None, None

### 6b. `torch.compile` on GPU

In [ ]:
if torch.cuda.is_available():
    print("Training on GPU with torch.compile...")
    try:
        gpu_compile_time, gpu_compile_losses, _, _ = train_with_compile(
            X, y, device='cuda', n_epochs=200, lr=0.01
        )
        print(f"\nGPU + torch.compile training time: {gpu_compile_time:.4f} seconds")
        print(f"Final loss: {gpu_compile_losses[-1]:.6f}")
    except Exception as e:
        print(f"torch.compile on GPU failed: {e}")
        gpu_compile_time, gpu_compile_losses = None, None
else:
    gpu_compile_time, gpu_compile_losses = None, None
    print("CUDA not available — skipping GPU + torch.compile training.")

## 7. Compare Running Times

In [ ]:
print("=" * 60)
print("              Running Time Comparison")
print("=" * 60)
print(f"CPU (eager)              : {cpu_time:.4f} s")
if gpu_time is not None:
    print(f"GPU (eager)              : {gpu_time:.4f} s")
if cpu_compile_time is not None:
    print(f"CPU + torch.compile      : {cpu_compile_time:.4f} s")
if gpu_compile_time is not None:
    print(f"GPU + torch.compile      : {gpu_compile_time:.4f} s")
print("=" * 60)

# Speedups vs CPU eager (the baseline)
print("\nSpeedups relative to CPU (eager):")
print(f"  CPU (eager)              : 1.00x  (baseline)")
if gpu_time is not None:
    print(f"  GPU (eager)              : {cpu_time/gpu_time:.2f}x")
if cpu_compile_time is not None:
    print(f"  CPU + torch.compile      : {cpu_time/cpu_compile_time:.2f}x")
if gpu_compile_time is not None:
    print(f"  GPU + torch.compile      : {cpu_time/gpu_compile_time:.2f}x")

## 8. Visualize Results

### 8a. Loss curves
All four configurations should converge to the same (near-zero) loss — the math is identical, only the execution differs.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(cpu_losses, label='CPU (eager)', linewidth=2)
if gpu_losses is not None:
    plt.plot(gpu_losses, label='GPU (eager)', linewidth=2, linestyle='--')
if cpu_compile_losses is not None:
    plt.plot(cpu_compile_losses, label='CPU + compile', linewidth=2, linestyle=':')
if gpu_compile_losses is not None:
    plt.plot(gpu_compile_losses, label='GPU + compile', linewidth=2, linestyle='-.')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Loss Curve')
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 8b. Running-time bar chart

In [ ]:
labels = ['CPU\n(eager)']
times  = [cpu_time]
colors = ['#4C72B0']

if gpu_time is not None:
    labels.append('GPU\n(eager)')
    times.append(gpu_time)
    colors.append('#DD8452')
if cpu_compile_time is not None:
    labels.append('CPU\n+ compile')
    times.append(cpu_compile_time)
    colors.append('#55A868')
if gpu_compile_time is not None:
    labels.append('GPU\n+ compile')
    times.append(gpu_compile_time)
    colors.append('#C44E52')

plt.figure(figsize=(9, 5))
bars = plt.bar(labels, times, color=colors)
plt.ylabel('Time (seconds)')
plt.title('Training Time: CPU vs GPU vs torch.compile')
for bar, t in zip(bars, times):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f'{t:.3f} s', ha='center', va='bottom', fontsize=10)
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Sanity Check: Did We Recover the True Parameters?

In [ ]:
# Bring CPU result back to host (it already is) and compare with the true weights
w_learned = w_cpu.cpu().flatten()
w_true    = true_w.flatten()

w_error = (w_learned - w_true).abs().mean().item()
b_error = (b_cpu.cpu() - true_b).abs().item()

print(f"Mean absolute error on w : {w_error:.6f}")
print(f"Absolute error on bias b : {b_error:.6f}")
print(f"True bias  = {true_b.item():.4f}")
print(f"Learned b  = {b_cpu.item():.4f}")

# Plot first 30 weights — learned vs true
plt.figure(figsize=(10, 4))
idx = np.arange(30)
plt.plot(idx, w_true[:30].numpy(),    'o-', label='true w',    markersize=8)
plt.plot(idx, w_learned[:30].numpy(), 'x--', label='learned w', markersize=8)
plt.xlabel('Feature index')
plt.ylabel('Weight value')
plt.title('Learned vs True Weights (first 30 dimensions)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Discussion

### What did we observe?
- **All configurations converge to the same loss** — autograd produces identical gradients regardless of where or how we execute the graph.
- The **GPU is typically faster** than CPU for this problem because each epoch performs a large matrix multiplication (`X @ w` with shape `100,000 × 500`), which the GPU parallelizes massively.
- **`torch.compile` adds further speedup** by fusing operations into single optimized kernels. The win is usually **larger on GPU** because more ops can be fused into one kernel launch — kernel-launch overhead is significant on GPU but negligible on CPU.
- For **very small tensors** the GPU (and `torch.compile`) can actually be *slower* than eager CPU, because every kernel launch has fixed overhead and data must be copied across the PCIe bus.

### Why we synchronize on GPU
CUDA kernels are launched **asynchronously** — Python returns immediately while the GPU is still computing. If we time naively, we'd measure only the launch overhead, not the actual computation. `torch.cuda.synchronize()` blocks until all queued work is done, giving us a fair measurement.

### Why we warm up (twice!)
1. **CUDA warm-up**: the first CUDA call in a session triggers context creation, kernel compilation, and memory-allocator setup.
2. **`torch.compile` warm-up**: the *first* invocation of a compiled function triggers the JIT compilation itself, which can take several seconds. Including that in the timing would make `torch.compile` look much slower than it really is during steady-state training.

### When `torch.compile` helps most
- Many small ops that can be fused (element-wise chains, normalization layers, activation + matmul patterns).
- Long-running training loops where the compile cost is amortized over many steps.
- GPU with large enough work per step that kernel-launch overhead matters.

### When `torch.compile` may *not* help
- Very short runs where compile time dominates.
- Pure single-matmul workloads on CPU (the matmul is already calling a highly optimized BLAS kernel — there's nothing left to fuse).
- Code with frequent shape changes or Python-side control flow that triggers recompilation.

### Things to try
1. Increase `N_SAMPLES` and `N_FEATURES` — does the GPU + compile speedup grow?
2. Decrease them dramatically (e.g. 100 × 5) — does eager CPU win?
3. Try `torch.compile(forward, mode="reduce-overhead")` or `mode="max-autotune"` and compare.
4. Replace the manual update with `torch.optim.SGD` and confirm you get the same result.
5. Switch to mini-batch gradient descent and see how the timing changes.